In [ ]:
!pip install pyspark

In [ ]:
!wget -O owid-covid-data.csv https://raw.githubusercontent.com/owid/covid-19-data/master/public/data/owid-covid-data.csv

In [8]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, sum, desc

In [9]:
spark = SparkSession.builder \
    .appName("COVID19 Data Analysis") \
    .getOrCreate()

In [ ]:
df = spark.read.csv("owid-covid-data.csv", header=True, inferSchema=True)

df.show(5)
df.printSchema()

In [ ]:
df.select("location", "date", "total_cases", "total_deaths").show(20)

In [18]:
df_valid = df.filter(col("total_cases") > 0)

In [19]:
df_clean = df_valid.select(
    "location",
    "date",
    "total_cases",
    "total_deaths"
).dropna()

In [ ]:
pakistan_data = df_clean.filter(col("location") == "Pakistan")
pakistan_data.show()

In [ ]:
df_clean.select(
    sum("total_cases").alias("Total Cases"),
    sum("total_deaths").alias("Total Deaths")
).show()

In [ ]:
top_countries = df_clean.groupBy("location") \
    .sum("total_cases") \
    .orderBy(desc("sum(total_cases)"))

top_countries.show(10)

In [23]:
spark.stop()